In [0]:
name: DABs Production CI/CD Pipeline
on:
  pull_request:
    branches:
      - main
  push:
    branches:
      - '**'
jobs:
  test-and-deploy:
    runs-on: ubuntu-latest
    env:
      DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
      DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_USER_TOKEN }}
    steps:
      - name: Checkout Code
        uses: actions/checkout@v3

      - name: Setup JDK 11
        uses: actions/setup-java@v3
        with:
          java-version: '11'
          distribution: 'temurin'

      - name: Setup Python 3.10
        uses: actions/setup-python@v4
        with:
          python-version: '3.10'
      
      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Build Local Packages
        run: |
          python -m build logic_packages/

      - name: Install Local Dependencies
        run: |
          pip3 install logic_packages/dist/*.whl

      - name: Run Unit Tests
        run: |
          export PYTHONPATH="$PWD"
          python -m unittest tests/*.py

      - name: Install Databricks CLI
        run: |
          curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh

      - name: Validate Bundle
        run: databricks bundle validate

      - name: Deploy Bundle Production
        if: github.event_name == 'push' && github.ref == 'refs/heads/main'
        run: databricks bundle deploy -t prod

      - name: Deploy Bundle Non-Production
        if: github.ref != 'refs/heads/main'
        run: databricks bundle deploy -t dev